In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
import torch
from transformers import pipeline

C:\Users\micha\anaconda3\envs\transformers\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pipe_base = pipeline(
    "text-generation",
    model="openai-community/gpt2",
    device="cuda",
)

prompt = "Once upon a time there was a fairy"

baseline = pipe_base(
    prompt,
    max_new_tokens=200,
    min_new_tokens=100,
    do_sample=True,  # enables random sampling
    temperature=0.9, # controls randomness: small is deterministic, repetitive, high is chaotic, creative, 1.0 is standard. 0.9 = slightly creative but still coherent
    top_p=0.95       # nucleus sampling 0.95: prevents extremely unlikely words from being chosen
    repetition_penalty=1.1, # penalizes tokens that have already appeared: >1.0 → discourage repetition
    no_repeat_ngram_size=3 # prevents repeating any 3-word sequence exactly
)

baseline = baseline[0]["generated_text"] # extracts the generated text
print(baseline)

Device set to use cuda
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Once upon a time there was a fairy called Euphemia, the witch who was able to get people to come to the Wizard Academy, and to know about it. That Fairy knew about the magic of magic.

"Why, you're so curious? If you'd just come to learn more about this fairy, I'd've told you about this Fairy. There were plenty of times when I thought that fairy would understand magic. I never saw her being quite that way, and I never saw her having any trouble understanding magic, either."

"Oh, I see."

"Did you ever wonder how some magical beings of the past lived?"

"I haven't been able to find out. I'd have to go in and try and figure out what they were. Maybe I should explain this to your father. He would be much more interesting than any human would be for you. I won't go on if I think you're stupid. At the very least, if you're not interested in studying


In [3]:
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer

# Load dataset
dataset = load_dataset('vicclab/fairy_tales')

In [4]:
train_val = dataset["train"].train_test_split(
    test_size=0.2, seed=42)

In [5]:
dataset = DatasetDict({
    "train": train_val["train"],
    "validation": train_val["test"]
})

In [6]:
print("Train size:", len(dataset["train"]))
print("Validation size:", len(dataset["validation"]))

Train size: 82878
Validation size: 20720


In [7]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained('openai-community/gpt2')
tokenizer.pad_token = tokenizer.eos_token

In [8]:
def tokenize_function(examples):
    """
    Takes raw text (strings), converts it into model-readable tokens (integers), 
    enforces a maximum sequence length, returns a dictionary suitable for model input
    """
    enc = tokenizer(
        examples["text"],
        truncation=True,
        max_length=256   # ↑ important: longer context
    )
    return enc

In [9]:
tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

In [10]:
tokenized_datasets = tokenized_datasets.filter(
    lambda x: len(x["input_ids"]) > 0
)

In [11]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [12]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

model = AutoModelForCausalLM.from_pretrained(
    "openai-community/gpt2"
).to("cuda")

training_args = TrainingArguments(
    output_dir="models/results",
    eval_strategy="epoch",
    num_train_epochs=5,              # ↑ more signal at least 5
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_dir="models/logs",
    save_strategy="epoch",
    report_to="none"
)

In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
)

trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.412700,3.552787
2,3.177700,3.476668
3,2.904400,3.489201
4,2.717600,3.526197
5,2.532700,3.583601


TrainOutput(global_step=83275, training_loss=2.977680019079762, metrics={'train_runtime': 10073.2578, 'train_samples_per_second': 33.067, 'train_steps_per_second': 8.267, 'total_flos': 3373994603520000.0, 'train_loss': 2.977680019079762, 'epoch': 5.0})

---

In [13]:
checkpoint_path_16655 = "models/results/checkpoint-16655"

In [14]:
tokenizer_16655 = AutoTokenizer.from_pretrained(checkpoint_path_16655)
model_16655 = AutoModelForCausalLM.from_pretrained(checkpoint_path_16655).to("cuda")

In [15]:
pipe_ckpt = pipeline(
    "text-generation",
    model=model_16655,
    tokenizer=tokenizer_16655,
    device="cuda"
)

Device set to use cuda


In [16]:
ft_output_16655 = pipe_ckpt(
    prompt,
    max_new_tokens=200,
    min_new_tokens=100,
    do_sample=True,
    temperature=0.9,
    top_p=0.95,
    repetition_penalty=1.1,
    no_repeat_ngram_size=3
)

finetuned_16655 = ft_output_16655[0]["generated_text"]
print(finetuned_16655)

Once upon a time there was a fairy, who lived on her throne. At last they heard the king of all the fairies, and she said she would like to go up to the mountain, but it was not a good idea. So, if he could get out of it he had to run in, and he might as well try. At the top of the mountain was a great rock, that stood close by, but one day she went to the top, and when the father of the fair people wanted to get out it, he couldn’t; and so she went up and up the hill, and then she came to the old man again. So she took him up, and bade him let a golden feather fly away, and in a minute he was standing before the door. Now she bade his son go into the hall, and now the King was up on his knees, and so fast he was that he got very angry at him and said to him: "It is an easy thing to


---

In [17]:
checkpoint_path_33310 = "models/results/checkpoint-33310"

In [18]:
tokenizer_33310 = AutoTokenizer.from_pretrained(checkpoint_path_33310)
model_33310 = AutoModelForCausalLM.from_pretrained(checkpoint_path_33310).to("cuda")

In [19]:
pipe_ckpt = pipeline(
    "text-generation",
    model=model_33310,
    tokenizer=tokenizer_33310,
    device="cuda"
)

Device set to use cuda


In [20]:
ft_output_33310 = pipe_ckpt(
    prompt,
    max_new_tokens=200,
    min_new_tokens=100,
    do_sample=True,
    temperature=0.9,
    top_p=0.95,
    repetition_penalty=1.1,
    no_repeat_ngram_size=3
)

finetuned_33310 = ft_output_33310[0]["generated_text"]
print(finetuned_33310)

Once upon a time there was a fairy who had three children, one with a long nose and another with a very long chin. The son of these were called Beauty, but she had two eyes, and a very big mouth. Her name was Gold-skin. The mother loved to have her own daughter in her. She was so good, and Gold-red. So he said he would have her, and she said, but he should have her and all his money, but, as for the golden one, and the other; the gold and silver, and it fell, and he thought, she must get him, and then, if she were not dead; she must go and have them. So, he made up his mind to do it, and that night he found a big hill, and just when he went in he met a man. He did not find her, but the woman took care and put him under, and sat him down on a green patch. And when he got into it, a man came


---

In [21]:
checkpoint_path_49965 = "models/results/checkpoint-49965"

In [22]:
tokenizer_49965 = AutoTokenizer.from_pretrained(checkpoint_path_49965)
model_49965 = AutoModelForCausalLM.from_pretrained(checkpoint_path_49965).to("cuda")

In [23]:
pipe_ckpt = pipeline(
    "text-generation",
    model=model_49965,
    tokenizer=tokenizer_49965,
    device="cuda"
)

Device set to use cuda


In [24]:
ft_output_49965 = pipe_ckpt(
    prompt,
    max_new_tokens=200,
    min_new_tokens=100,
    do_sample=True,
    temperature=0.9,
    top_p=0.95,
    repetition_penalty=1.1,
    no_repeat_ngram_size=3
)

finetuned_49965 = ft_output_49965[0]["generated_text"]
print(finetuned_49965)

Once upon a time there was a fairy princess who had three sons. The eldest of the three was called Nouronnihar, and his father and mother were both tall and fair; but they were afraid of him, and he loved them so much that they could not live, and so they said they should not go out into the world, and keep their own way, and did. So he wanted to have a little boy, but would rather go and seek the land. But she told him what he could do, and what he liked best. Then, when he had got to the top of the hill he said, “You shall never be able to tell me if you loved me, but let me go.” And as he walked on, Nouronicus noticed that the eldest had something wrong with the tree, for he loved it so very much, and when he got nearer to the edge of the rock he saw in front of him a beautiful little girl. But when he lifted up his eyes,


---

In [25]:
checkpoint_path_66620 = "models/results/checkpoint-66620"

In [26]:
tokenizer_66620 = AutoTokenizer.from_pretrained(checkpoint_path_66620)
model_66620 = AutoModelForCausalLM.from_pretrained(checkpoint_path_66620).to("cuda")

In [27]:
pipe_ckpt = pipeline(
    "text-generation",
    model=model_66620,
    tokenizer=tokenizer_66620,
    device="cuda"
)

Device set to use cuda


In [28]:
ft_output_66620 = pipe_ckpt(
    prompt,
    max_new_tokens=200,
    min_new_tokens=100,
    do_sample=True,
    temperature=0.9,
    top_p=0.95,
    repetition_penalty=1.1,
    no_repeat_ngram_size=3
)

finetuned_66620 = ft_output_66620[0]["generated_text"]
print(finetuned_66620)

Once upon a time there was a fairy called Niflheim, who had an only son. One day, when the mother was tired of her work at the castle and wished to rest some time in the garden she went to bed, but she could not get out of bed; so she thought her husband would go and try, and he would have to go, and the young man did; so they sat down by the well, and she began to scuttle about, and pick flowers, and when night came, and as soon as she was well out of it, she said to her husband: "Why don't you take away the bird, and I will let you stay at home." So they all went out again and sat down on the green path to the palace-yard, and Niflor grew up and fell asleep, and in spite of her good husband's bad weather, the sun was still shining, and after a while the boy went out into the garden, and there came a great bird which took


---

In [29]:
checkpoint_path_83275 = "models/results/checkpoint-83275"

In [30]:
tokenizer_83275 = AutoTokenizer.from_pretrained(checkpoint_path_83275)
model_83275 = AutoModelForCausalLM.from_pretrained(checkpoint_path_83275).to("cuda")

In [31]:
pipe_ckpt = pipeline(
    "text-generation",
    model=model_83275,
    tokenizer=tokenizer_83275,
    device="cuda"
)

Device set to use cuda


In [32]:
ft_output_83275 = pipe_ckpt(
    prompt,
    max_new_tokens=200,
    min_new_tokens=100,
    do_sample=True,
    temperature=0.9,
    top_p=0.95,
    repetition_penalty=1.1,
    no_repeat_ngram_size=3
)

finetuned_83275 = ft_output_83275[0]["generated_text"]
print(finetuned_83275)

Once upon a time there was a fairy who dwelt all alone in the middle of a lake, and she had a pretty little daughter. She was so small, and tiny, that her mother and sister couldn’t hear her. The fairy had a word to say about her; but when they were grown, the king said, she was, and he said, as she was too big. So the king set out with his youngest son to try if he would, and have her; so he went, and got the youngest, and the other, and set off. So when he came to the lake, the princess was so afraid that the wind and sun, and moon, and stars, and heaven, and earth, and sky, and that had two sons, and all the beasts, they had no fear when they saw the lad. So he did go and look, and looked at the lad, and then he looked, and there were the three sons. And now the king had two daughters, and these were


---

## Lexical Diversity

Higher = more varied phrasing

Distinct-n measures lexical diversity by computing the proportion of unique n-word sequences (n-grams) in a text, where higher values indicate less repetitive phrasing but do not capture semantic quality or true creativity, making it most useful for relative comparison (e.g., before vs. after fine-tuning) on texts of similar length.

In [33]:
def distinct_n(text, n=2):
    tokens = text.lower().split()
    ngrams = list(zip(*[tokens[i:] for i in range(n)]))
    return len(set(ngrams)) / max(1, len(ngrams))

In [34]:
base_text = baseline #baseline[0]["generated_text"]
ft_text_16655 = finetuned_16655 #ft_output[0]["generated_text"]
ft_text_33310 = finetuned_33310 #ft_output[0]["generated_text"]
ft_text_49965 = finetuned_49965 #ft_output[0]["generated_text"]
ft_text_66620 = finetuned_66620 #ft_output[0]["generated_text"]
ft_text_83275 = finetuned_83275 #ft_output[0]["generated_text"]

print("Distinct-2 (base):", distinct_n(base_text, 2))
print("Distinct-2 (ft):  ", distinct_n(ft_text_16655, 2))
print("Distinct-2 (ft):  ", distinct_n(ft_text_33310, 2))
print("Distinct-2 (ft):  ", distinct_n(ft_text_49965, 2))
print("Distinct-2 (ft):  ", distinct_n(ft_text_66620, 2))
print("Distinct-2 (ft):  ", distinct_n(ft_text_83275, 2))

Distinct-2 (base): 0.950920245398773
Distinct-2 (ft):   0.9441340782122905
Distinct-2 (ft):   0.9710982658959537
Distinct-2 (ft):   0.9542857142857143
Distinct-2 (ft):   0.9712643678160919
Distinct-2 (ft):   0.9532163742690059


---

## Surprisal (novelty vs generic English)

Slightly higher surprisal after fine-tuning = more novelty  
Too high = incoherent

This code measures surprisal by computing the model’s average negative log-likelihood (cross-entropy loss) over all tokens in the given text, which reflects how unexpected the text is to the model: higher loss means the model assigns lower probability to the observed tokens (higher surprisal), while lower loss means the text is more predictable according to the model.

In [35]:
def surprisal(model, tokenizer, text):
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        loss = model(**inputs, labels=inputs["input_ids"]).loss
    return loss.item()

base_surprisal = surprisal(pipe_base.model, pipe_base.tokenizer, base_text)
ft_surprisal_16655   = surprisal(pipe_base.model, pipe_base.tokenizer, ft_text_16655)
ft_surprisal_33310   = surprisal(pipe_base.model, pipe_base.tokenizer, ft_text_33310)
ft_surprisal_49965   = surprisal(pipe_base.model, pipe_base.tokenizer, ft_text_49965)
ft_surprisal_66620   = surprisal(pipe_base.model, pipe_base.tokenizer, ft_text_66620)
ft_surprisal_83275   = surprisal(pipe_base.model, pipe_base.tokenizer, ft_text_83275)

print("Surprisal (base):", base_surprisal)
print("Surprisal (ft):  ", ft_surprisal_16655)
print("Surprisal (ft):  ", ft_surprisal_33310)
print("Surprisal (ft):  ", ft_surprisal_49965)
print("Surprisal (ft):  ", ft_surprisal_66620)
print("Surprisal (ft):  ", ft_surprisal_83275)

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Surprisal (base): 2.4469382762908936
Surprisal (ft):   2.896608591079712
Surprisal (ft):   3.0161285400390625
Surprisal (ft):   2.904358148574829
Surprisal (ft):   2.681807041168213
Surprisal (ft):   2.7987918853759766


---

## Style alignment (fairy-tale vocabulary)

This code measures style alignment by counting how many distinct, predefined fairy-tale–related keywords appear in the text, using their presence as a simple proxy for how closely the text matches a fairy-tale style, without considering context, frequency, or deeper narrative structure.

In [36]:
fairy_words = {"fairy", "king", "queen", "magic", "forest", "spell", "castle"}

def fairy_score(text):
    tokens = set(text.lower().split())
    return len(tokens & fairy_words)

print("Fairy score (base):", fairy_score(base_text))
print("Fairy score (ft):  ", fairy_score(ft_text_16655))
print("Fairy score (ft):  ", fairy_score(ft_text_33310))
print("Fairy score (ft):  ", fairy_score(ft_text_49965))
print("Fairy score (ft):  ", fairy_score(ft_text_66620))
print("Fairy score (ft):  ", fairy_score(ft_text_83275))

Fairy score (base): 2
Fairy score (ft):   1
Fairy score (ft):   1
Fairy score (ft):   1
Fairy score (ft):   2
Fairy score (ft):   2


---

## Self-BLEU

Self-BLEU measures how similar generated samples are to each other. Lower Self-BLEU is better for creativity/diversity 

In [37]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

In [38]:
def self_bleu(texts, n_gram=2):
    """
    texts: list of generated strings
    n_gram: BLEU-n (2 or 3 recommended)
    """
    smoothie = SmoothingFunction().method1
    scores = []

    weights = {
        2: (0.5, 0.5),
        3: (1/3, 1/3, 1/3),
        4: (0.25, 0.25, 0.25, 0.25)
    }[n_gram]

    tokenized = [t.lower().split() for t in texts]

    for i, hypothesis in enumerate(tokenized):
        references = tokenized[:i] + tokenized[i+1:]
        score = sentence_bleu(
            references,
            hypothesis,
            weights=weights,
            smoothing_function=smoothie
        )
        scores.append(score)

    return sum(scores) / len(scores)

In [39]:
def generate_samples(pipe, prompt, n=30):
    return [
        pipe(
            prompt,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.9,
            top_p=0.95
        )[0]["generated_text"]
        for _ in range(n)
    ]

In [40]:
pipe_16655 = pipeline(
    "text-generation",
    model=model_16655,
    tokenizer=tokenizer_16655,
    device="cuda"
)

Device set to use cuda


In [44]:
baseline_texts = generate_samples(pipe_base, prompt, n=2)
finetuned_texts = generate_samples(pipe_16655, prompt, n=2)

print("Baseline Self-BLEU:", self_bleu(baseline_texts, n_gram=3))
print("Finetuned Self-BLEU:", self_bleu(finetuned_texts, n_gram=3))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Baseline Self-BLEU: 0.07240988286275696
Finetuned Self-BLEU: 0.1425921922408689


In [45]:
pipe_33310 = pipeline(
    "text-generation",
    model=model_33310,
    tokenizer=tokenizer_33310,
    device="cuda"
)

Device set to use cuda


In [46]:
baseline_texts = generate_samples(pipe_base, prompt, n=2)
finetuned_texts = generate_samples(pipe_33310, prompt, n=2)

print("Baseline Self-BLEU:", self_bleu(baseline_texts, n_gram=3))
print("Finetuned Self-BLEU:", self_bleu(finetuned_texts, n_gram=3))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Baseline Self-BLEU: 0.02765240737737341
Finetuned Self-BLEU: 0.13602801773284295


In [47]:
pipe_49965 = pipeline(
    "text-generation",
    model=model_49965,
    tokenizer=tokenizer_49965,
    device="cuda"
)

Device set to use cuda


In [48]:
baseline_texts = generate_samples(pipe_base, prompt, n=2)
finetuned_texts = generate_samples(pipe_49965, prompt, n=2)

print("Baseline Self-BLEU:", self_bleu(baseline_texts, n_gram=3))
print("Finetuned Self-BLEU:", self_bleu(finetuned_texts, n_gram=3))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Baseline Self-BLEU: 0.07829233221532633
Finetuned Self-BLEU: 0.11000107737948125


In [49]:
pipe_66620 = pipeline(
    "text-generation",
    model=model_66620,
    tokenizer=tokenizer_66620,
    device="cuda"
)

Device set to use cuda


In [50]:
baseline_texts = generate_samples(pipe_base, prompt, n=2)
finetuned_texts = generate_samples(pipe_66620, prompt, n=2)

print("Baseline Self-BLEU:", self_bleu(baseline_texts, n_gram=3))
print("Finetuned Self-BLEU:", self_bleu(finetuned_texts, n_gram=3))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Baseline Self-BLEU: 0.06872452628154654
Finetuned Self-BLEU: 0.12723215244587277


In [51]:
pipe_83275 = pipeline(
    "text-generation",
    model=model_83275,
    tokenizer=tokenizer_83275,
    device="cuda"
)

Device set to use cuda


In [52]:
baseline_texts = generate_samples(pipe_base, prompt, n=2)
finetuned_texts = generate_samples(pipe_83275, prompt, n=2)

print("Baseline Self-BLEU:", self_bleu(baseline_texts, n_gram=3))
print("Finetuned Self-BLEU:", self_bleu(finetuned_texts, n_gram=3))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Baseline Self-BLEU: 0.06360067533489525
Finetuned Self-BLEU: 0.1295260473053017


---

# Confirm by Upscaling

In [55]:
collection_base = []
for i in range(30):
    baseline = pipe_base(
        prompt,
        max_new_tokens=200,
        min_new_tokens=100,
        do_sample=True,
        temperature=0.9,
        top_p=0.95
    )
    collection_base.append(baseline[0]["generated_text"])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end gene

In [56]:
collection_base

['Once upon a time there was a fairy that sat on the edge of a river. It did not grow. It was only because the prince of the gods was strong that his mother knew his name.\n\nBut if he had said it that way, then the fairy would have remembered his name well.\n\nAt the age of six, the prince of the gods was summoned to the throne by his godess. He was the last of the king\'s children, and this made him strong enough to inherit the kingdom of the gods.\n\nThe prince of the gods spoke of the fairy and the prince of the gods. Their two names combined became the prince\'s and the prince\'s mother\'s name.\n\nAs the prince spoke, the prince\'s mother spoke softly and slowly.\n\n"My name is Kukuru-sama."\n\nThe fairy smiled softly, turning her head in front of the prince.\n\n"Now that I think about it, I think there is a fairy in the heavens.\n\n',
 'Once upon a time there was a fairy called Arthmar who used to talk to people of all ages and shapes. This fairy lived in the castle of Hildheim 

In [57]:
collection_16655 = []
for i in range(30):
    ft_output_16655 = pipe_ckpt(
    prompt,
    max_new_tokens=200,
    min_new_tokens=100,
    do_sample=True,
    temperature=0.9,
    top_p=0.95,
    repetition_penalty=1.1,
    no_repeat_ngram_size=3
    )
    collection_16655.append(ft_output_16655[0]["generated_text"])

In [58]:
collection_16655

['Once upon a time there was a fairy queen who had a very pretty child. The baby was so tiny, that her father and mother could not bear to see her; so they took her up to the well, but did all she could to look into the wall, but, and she did not. When the girl was grown, she got in; and it became the custom, that every one of them put a little silver spoon into the water, and when it was about a foot deep. She began to eat out of it, and then one of the boys said: "Oh, if you like, mother, that\'s so small, I mayn\'t mind, and I\'ll go in." And she went in and filled the pot with water, but as she came down, she forgot her spoon, and the water and salt, and forgot the spoon, for she was no more. And the night after this, on a spring day, poor old Ritter Red crept into the well and ate some of the gold that',
 'Once upon a time there was a fairy who lived in the forest; and while she did not like having to go about alone in her wood, she had many other plans. One night, when she was ve

In [59]:
collection_33310 = []
for i in range(30):
    ft_output_33310 = pipe_ckpt(
    prompt,
    max_new_tokens=200,
    min_new_tokens=100,
    do_sample=True,
    temperature=0.9,
    top_p=0.95,
    repetition_penalty=1.1,
    no_repeat_ngram_size=3
    )
    collection_33310.append(ft_output_33310[0]["generated_text"])

In [60]:
collection_33310

["Once upon a time there was a fairy, who had two daughters; one very beautiful and the other very ugly. They were both so pretty that she had no peace of mind, but she loved them dearly. The young ones thought that they would be good, and the old one was to look after them, and did all her best, and when the little one was more, and took care of them. One day, when they were all: she set out on her journey, she bade the two little one, and put her foot into the basket and took it to the king’s palace, so that the rest would not know that he had a son. So when she had set off for the king's castle, and got home, she said to him, ‘I will give you a piece of advice: if the prince had but one daughter, she might get rich fast. If you take care of her, it shall soon come to pass.’ But the king thought, and said that he would",
 'Once upon a time there was a fairy queen who had seven children. Each of them was very beautiful, but as each daughter was young she grew more beautiful and more b

In [61]:
collection_49965 = []
for i in range(30):
    ft_output_49965 = pipe_ckpt(
    prompt,
    max_new_tokens=200,
    min_new_tokens=100,
    do_sample=True,
    temperature=0.9,
    top_p=0.95,
    repetition_penalty=1.1,
    no_repeat_ngram_size=3
    )
    collection_49965.append(ft_output_49965[0]["generated_text"])

In [62]:
collection_49965

['Once upon a time there was a fairy woman who lived in the enchanted forest and had a daughter, who had only one eye. When she was seven years old, her mother and father passed away and she was left alone in the woods. The mother and the children, when they were grown up, and were left alone with their mother, thought of going to some lonely spot, came near the mountains. They took a long walk that was so far away, they were too tired to get up, but they could not find the way, but that, and they did, and she set out into it. But as they walked through it, a little way, and there, they met a bear and a man. When the day grew old, and the bear was out on the high road, she got up, took a sledge, and set out on foot. She walked up and walked for several days, but he never came back, and all the while she seemed very hungry, and was thinking of leaving her. The bear',
 "Once upon a time there was a fairy that lived among the hills and valleys, and that had a wife and children. The fairy 

In [63]:
collection_66620 = []
for i in range(30):
    ft_output_66620 = pipe_ckpt(
    prompt,
    max_new_tokens=200,
    min_new_tokens=100,
    do_sample=True,
    temperature=0.9,
    top_p=0.95,
    repetition_penalty=1.1,
    no_repeat_ngram_size=3
    )
    collection_66620.append(ft_output_66620[0]["generated_text"])

In [64]:
collection_66620

['Once upon a time there was a fairy queen who had a son and wished to have her son. The king had no daughter, so he sent for his princesses; but when they were out hunting, one of them noticed that they were not long in getting into the water, and when he went to look in. So, when he came to the shore, he found the water-lily, and as he was going, he saw a silver fish. When he looked in the water once more, he got in, and, his father, his old dame, who lived in the other side, said: ‘That is not such a husband you may be in all my days, I think,’ and he said, ‘I’d better try it, for I’m a good man.” So she put a gold-fish in her mouth, and let him go, just as she did, without finding any one. So the son took the silver-fish, and away',
 'Once upon a time there was a fairy who lived a long way off in the wood. One day, when she was in her natural shape, she met a little blue-eyed boy, and as she looked very sad, he began to cry, and to think she must be dead. So they called the boy. Ho

In [65]:
collection_83275 = []
for i in range(30):
    ft_output_83275 = pipe_ckpt(
    prompt,
    max_new_tokens=200,
    min_new_tokens=100,
    do_sample=True,
    temperature=0.9,
    top_p=0.95,
    repetition_penalty=1.1,
    no_repeat_ngram_size=3
    )
    collection_83275.append(ft_output_83275[0]["generated_text"])

In [66]:
collection_83275

['Once upon a time there was a fairy who had three sons. All the princes and great kings had heard that, so they had set out to find out how the four-footed little man got to walk, and, and when they came to the castle, they said they would give him the youngest son; and he. The first son was named Peter, but he was the youngest. The second son was Tito. The third, the same name, because he was called Walter, and because, he was a boy, and the rest were all at ease with him: his father, for he was always good to have, but for each of them he liked nothing better, and wished to know more. They took him and gave him until he was twenty years old; and when all three died, she made him her own child. So when the third son was born, the king, who loved him very much, but did not know how to rid himself after a long absence from the royal family, and told his mother all',
 'Once upon a time there was a fairy, who had an only son and one daughter. The son was called Käthchen, but his mother o

---

## Lexical Diversity

In [67]:
def distinct_n(text, n=2):
    tokens = text.lower().split()
    ngrams = list(zip(*[tokens[i:] for i in range(n)]))
    return len(set(ngrams)) / max(1, len(ngrams))

In [68]:
print("Distinct-2 (base):", distinct_n(collection_base[0], 2))
print("Distinct-2 (ft):  ", distinct_n(collection_16655[0], 2))
print("Distinct-2 (ft):  ", distinct_n(collection_33310[0], 2))
print("Distinct-2 (ft):  ", distinct_n(collection_49965[0], 2))
print("Distinct-2 (ft):  ", distinct_n(collection_66620[0], 2))
print("Distinct-2 (ft):  ", distinct_n(collection_83275[0], 2))

Distinct-2 (base): 0.84472049689441
Distinct-2 (ft):   0.9653179190751445
Distinct-2 (ft):   0.9595375722543352
Distinct-2 (ft):   0.9382022471910112
Distinct-2 (ft):   0.9743589743589743
Distinct-2 (ft):   0.9548022598870056


---

## Surprisal (novelty vs generic English)

In [69]:
def surprisal(model, tokenizer, text):
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        loss = model(**inputs, labels=inputs["input_ids"]).loss
    return loss.item()

base_surprisal = surprisal(pipe_base.model, pipe_base.tokenizer, collection_base[0])
ft_surprisal_16655   = surprisal(pipe_base.model, pipe_base.tokenizer, collection_16655[0])

print("Surprisal (base):", base_surprisal)
print("Surprisal (ft):  ", ft_surprisal_16655)
print("Surprisal (ft):  ", ft_surprisal_33310)
print("Surprisal (ft):  ", ft_surprisal_49965)
print("Surprisal (ft):  ", ft_surprisal_66620)
print("Surprisal (ft):  ", ft_surprisal_83275)

Surprisal (base): 2.2117695808410645
Surprisal (ft):   2.9573049545288086
Surprisal (ft):   3.0161285400390625
Surprisal (ft):   2.904358148574829
Surprisal (ft):   2.681807041168213
Surprisal (ft):   2.7987918853759766


---

## Style alignment (fairy-tale vocabulary)

In [70]:
fairy_words = {"fairy", "king", "queen", "magic", "forest", "spell", "castle"}

def fairy_score(text):
    tokens = set(text.lower().split())
    return len(tokens & fairy_words)

print("Fairy score (base):", fairy_score(collection_base[0]))
print("Fairy score (ft):  ", fairy_score(collection_16655[0]))
print("Fairy score (ft):  ", fairy_score(collection_33310[0]))
print("Fairy score (ft):  ", fairy_score(collection_49965[0]))
print("Fairy score (ft):  ", fairy_score(collection_66620[0]))
print("Fairy score (ft):  ", fairy_score(collection_83275[0]))

Fairy score (base): 1
Fairy score (ft):   2
Fairy score (ft):   1
Fairy score (ft):   2
Fairy score (ft):   3
Fairy score (ft):   1


---

## Self-BLEU

In [71]:
baseline_texts = generate_samples(pipe_base, prompt, n=30)
finetuned_texts = generate_samples(pipe_16655, prompt, n=30)

print("Baseline Self-BLEU:", self_bleu(baseline_texts, n_gram=3))
print("Finetuned Self-BLEU:", self_bleu(finetuned_texts, n_gram=3))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end gene

Baseline Self-BLEU: 0.2578955831839715
Finetuned Self-BLEU: 0.4460245050680139


In [72]:
baseline_texts = generate_samples(pipe_base, prompt, n=30)
finetuned_texts = generate_samples(pipe_33310, prompt, n=30)

print("Baseline Self-BLEU:", self_bleu(baseline_texts, n_gram=3))
print("Finetuned Self-BLEU:", self_bleu(finetuned_texts, n_gram=3))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end gene

Baseline Self-BLEU: 0.2639484229390665
Finetuned Self-BLEU: 0.4858759726208421


In [73]:
baseline_texts = generate_samples(pipe_base, prompt, n=30)
finetuned_texts = generate_samples(pipe_49965, prompt, n=30)

print("Baseline Self-BLEU:", self_bleu(baseline_texts, n_gram=3))
print("Finetuned Self-BLEU:", self_bleu(finetuned_texts, n_gram=3))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end gene

Baseline Self-BLEU: 0.2769160329681222
Finetuned Self-BLEU: 0.4424985910297682


In [74]:
baseline_texts = generate_samples(pipe_base, prompt, n=30)
finetuned_texts = generate_samples(pipe_66620, prompt, n=30)

print("Baseline Self-BLEU:", self_bleu(baseline_texts, n_gram=3))
print("Finetuned Self-BLEU:", self_bleu(finetuned_texts, n_gram=3))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end gene

Baseline Self-BLEU: 0.25758882501283564
Finetuned Self-BLEU: 0.4634396084033125


In [75]:
baseline_texts = generate_samples(pipe_base, prompt, n=30)
finetuned_texts = generate_samples(pipe_83275, prompt, n=30)

print("Baseline Self-BLEU:", self_bleu(baseline_texts, n_gram=3))
print("Finetuned Self-BLEU:", self_bleu(finetuned_texts, n_gram=3))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end gene

Baseline Self-BLEU: 0.2714595230869056
Finetuned Self-BLEU: 0.42785731298245333
